# Reachability-based BLAs for intersection crossings

Companion to [protocol.ipynb](protocol.ipynb) for corridor-valued offers.

- Cells tagged `definitions` are the method; `run_scenario` compiles them for the serial runner.
- Kernel: Python 3.13, `pyspect[hj_reachability]==1.2`.
- Evidence: **[algebra]** normalization · **[semantic]** containment and fixed client policy ·
  **[numerical]** grid membership · **[independent]** analytic or nominal checks.
- Pipeline: contracts → owner obligation $F_s$ → generated query → TLT → HJ realization.
- Scope: point-mass kinematic bicycle, exact entry states, nominal HJ-feedback client,
  finite departure library. No robustness certificate.

In [ ]:
from dataclasses import replace
from itertools import combinations
from pathlib import Path
from IPython.display import Markdown, display
import os
import sys
import warnings
import numpy as np

warnings.filterwarnings('ignore', message='IProgress not found')  # tqdm without ipywidgets
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / 'src' / 'ltms_reachability').is_dir())
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
from scipy.interpolate import RegularGridInterpolator
from scipy.ndimage import distance_transform_edt, maximum_filter
from pyspect import Set
from pyspect.set_builder import BoxSet, Inter, PolytopeSet, Union
from pyspect.impls.hj_reachability import TVHJImpl
from pyspect.systems.hj_reachability import Bicycle4D
from ltms_reachability import LTMSPlanner, TimeWindow, WindowPolicy, create_4way
from ltms_bla.checks import boxes_covered, sampled_separation
from ltms_bla.tlt import compile_query
from ltms_bla.presentation import (MEANINGS, table, candidate_rows, results_table,
                                   plot_scenario, export_report)
from ltms_bla.records import SERVICE_CHECKS, implies, refines, ledger_symbols
from ltms_bla.scenario import file_digest, run_scenario, build_report, outcome_summary
from ltms_bla.static_figure import export_figure
from protocol.presentation import display_timeline

OUTPUT = Path(os.environ.get('LTMS_OUTPUT', ROOT / 'artifacts' / 'ltms_blas'))
OUTPUT.mkdir(parents=True, exist_ok=True)

In [ ]:
from dataclasses import asdict
from fractions import Fraction as F
import json
import numpy as np
import interval_logic as il
from protocol import Contract, CommitResult, affine_policies, normalize
from ltms_bla.bargaining import load_bargain
from ltms_bla.checks import aligned_disjoint
from ltms_bla.records import (CrossingService, CrossingRequest, Candidate, ServiceAdmission, Admission,
                              CorridorOffer, standing, snapshot_id, digest, certify_service,
                              certify_compatibility, commit_exact)
from ltms_bla.scenario import verify_artifact, write_artifact

# The protocol.ipynb function body; only its offer-eligibility validator differs.
bargain = load_bargain(ROOT)

## 1. Model

Owner controls $r_j(t)\subseteq\mathbb R^2$; client $j$ controls $u=(\delta,a)$.
$$\dot x=v\cos h,\quad \dot y=v\sin h,\quad \dot h=v\tan\delta/L,\quad \dot v=a .$$
Point vehicle at $(x,y)$; route-local chart, westbound = rotation by $\pi$.

In [ ]:
WHEELBASE = .32                                                                # L [m]
CONTROLS = dict(min_steer=-.15, max_steer=.15, min_accel=-.15, max_accel=.15)  # δ [rad], a [m/s²]
CHART = dict(h=(-.5, .5), v=(.3, .6))                                          # route-local chart
VMAX = CHART['v'][1]


def bicycle4d(z, u):
    x, y, h, v = z
    steer, accel = u
    return np.array([v*np.cos(h), v*np.sin(h), v*np.tan(steer)/WHEELBASE, accel])

Regions are boxes over named axes. $\mathit{road}_e=\texttt{LANE}\cup\text{octagon}$, zones $=\text{box}\cap\mathit{road}_e$.
Task: $\mathit{road}_e\ \mathcal U\ \mathit{exit}_e$. Pointwise checks use $\texttt{LANE}\subseteq\mathit{road}_e$ and
$\texttt{LANE}\cap\texttt{EXIT\_ZONE}\subseteq\mathit{exit}_e$.

In [ ]:
AXIS = dict(x=0, y=1, h=2, v=3)
SPEED_LIMIT = (.3, .6)
LANE = dict(y=(-.8, 0.), v=SPEED_LIMIT)                          # eastbound lane; its heading limit ±π/5 contains CHART['h']
MANAGED = 1.2                                                    # octagon within [-1.2, 1.2]^2
ENTRY_ZONE = dict(x=(-2., -1.4), y=(-.68, -.12))
EXIT_ZONE = dict(x=(1.4, 2.), y=(-.68, -.12))


def inside(z, *boxes):
    return all(lo <= z[AXIS[k]] <= hi for box in boxes for k, (lo, hi) in box.items())

**Task and capability.** Departure $s$ from `ENTRY` ($z(s)=z_0$); lease $[s,s+8]$.
$$\Phi_s:=(\mathit{road}\land\mathit{alloc}_j)\ \mathcal U_{[s,d_\theta]}\ \mathit{exit}\ \ (\text{read at }s),\qquad
T_\theta:=\bigvee_{s\in D_\theta}\Phi_s,$$
$$F_s(r_j):=\exists u(\cdot):\ z^{s,z_0}_u\vDash\Phi_s,\qquad S_a:=\bigvee_{s\in D_\theta}F_s .$$
- $\mathit{alloc}_j$: $p(z(t))\in r_j(t)$, the owner's variable. $F_s$ depends on $r_j$ only: owner-only.
- Client $\pi_r$: realize $V_r$ for $\Phi_s$ with $\mathit{alloc}:=r$; execute $u\in\arg\min_u\nabla V_r\cdot f(z,u)$.
  One rule, parameterized by the offered allocation.
- Departure: earliest $s$ whose $F_s$ the agreement implies.

In [ ]:
SPEED = .45
ENTRY = (-1.6, -.4, 0., SPEED)  # route-local (x, y, h, v) at departure
LEASE = 8                        # lease on [s, s+8]


def certified_window_applies(departure, deadline):
    # The certifying query uses the lease window [s, s+8] ⊆ [s, d]; then it implies F_s.
    return departure + LEASE <= deadline


def speed_bound_excludes(departure, deadline):
    # Independent necessary condition for F_s: the vehicle must travel from its entry to the exit zone.
    return F(str(VMAX))*(deadline - departure) < F(str(EXIT_ZONE['x'][0])) - F(str(ENTRY[0]))


def hj_feedback(gradient):
    # π_r: u ∈ argmin_u ∇V_r · f(z, u). f is affine in tan δ and a, so the minimizer is bang-bang.
    def policy(t, z):
        dh, dv = gradient(t, z)
        steer = CONTROLS['min_steer'] if dh > 0 else CONTROLS['max_steer'] if dh < 0 else 0.
        accel = CONTROLS['min_accel'] if dv > 0 else CONTROLS['max_accel'] if dv < 0 else 0.
        return steer, accel
    return policy


def execute(policy, start=ENTRY, until=LEASE, step=.025):
    # The client's execution from the entry state: RK4, input held over each step,
    # stopping once the position is in the exit zone (the until's witness).
    t, z = 0., np.array(start, float)
    times, states, inputs = [t], [z.copy()], []
    while t < until - 1e-9 and not inside(z, LANE, EXIT_ZONE):
        u = policy(t, z)
        k1 = bicycle4d(z, u)
        k2 = bicycle4d(z + step/2*k1, u)
        k3 = bicycle4d(z + step/2*k2, u)
        k4 = bicycle4d(z + step*k3, u)
        z, t = z + step/6*(k1 + 2*k2 + 2*k3 + k4), t + step
        times.append(t), states.append(z.copy()), inputs.append(u)
    return np.array(times), np.array(states), np.array(inputs)

| | Meaning | Actor | Evidence |
|:--|:--|:--|:--|
| $H_\kappa$ | $r_j\supseteq R_\kappa$ on $[s,s+8]$ | owner | promise; $R_\kappa$ from §2 |
| $F_s$ | $\exists u$: $\Phi_s$ under $r_j$ | owner | query generated from $F_s$ with $\mathit{alloc}:=R_\kappa$ (§2) |
| $S_a$ | $\bigvee F_s$ | owner | definition |
| $\Phi_s$ | $(\mathit{road}\land\mathit{alloc}_j)\,\mathcal U_{[s,d]}\,\mathit{exit}$ | client | $\pi_R$ executed and checked (§2) |

**[semantic]** $F_s(r)$ and $\pi_r$ give $\Phi_s$ (nominal dynamics, exact $V_r$), so $C_{\rm client}=\langle S_a\mid T_\theta\rangle$.
$\mathit{alloc}$ occurs only positively in $\Phi_s$, so $F_s$ is monotone in $r$: $H_\kappa\vDash_f F_s\iff F_s(R_\kappa)$.

## 2. Environment, task and corridor

Grid over $(t,x,y,h,v)$: 9 s in 0.25 s steps.

In [ ]:
HORIZON, DT = 9, .25
AXES = [dict(name='t', bounds=[0, HORIZON], points=round(HORIZON/DT) + 1),
        dict(name='x', bounds=[-2, 2], points=31), dict(name='y', bounds=[-2, 2], points=61),
        dict(name='h', bounds=list(CHART['h']), points=9), dict(name='v', bounds=list(CHART['v']), points=7)]
DYNAMICS = dict(cls=Bicycle4D, wheelbase=WHEELBASE, **CONTROLS)
ACCURACY = 'medium'

impl = TVHJImpl(dict(DYNAMICS), AXES, accuracy=ACCURACY)  # copy: TVHJImpl pops 'cls'
planner = LTMSPlanner(impl)
grid, times = impl.grid, planner.times
x, y, h, v = (np.asarray(c) for c in grid.coordinate_vectors)

The interpretation of §1's state atoms: pyspect set builders, realized on the grid (inside ⇔ $\le0$) and
checked against the legacy map.

In [ ]:
OCTAGON = PolytopeSet(normals=[[1, 1], [-1, 1], [-1, -1], [1, -1]],  # |x| + |y| ≤ 2
                      offsets=[[-1, -1], [1, -1], [1, 1], [-1, 1]], axes=['x', 'y'])
CENTRE = Inter(OCTAGON, BoxSet(dict(x=(-MANAGED, MANAGED), y=(-MANAGED, MANAGED), v=SPEED_LIMIT), axes=['x', 'y', 'v']))
ROAD_E = Union(BoxSet(LANE, axes=['y', 'v']), CENTRE)
ENTRY_W = Inter(BoxSet(ENTRY_ZONE, axes=['x', 'y']), ROAD_E)
EXIT_E = Inter(BoxSet(EXIT_ZONE, axes=['x', 'y']), ROAD_E)
LANE_STRIP = (-.46, -.34)  # the offered corridor keeps the vehicle in this strip of the lane
assert LANE['y'][0] <= LANE_STRIP[0] and LANE_STRIP[1] <= LANE['y'][1]

road_e, entry_w, exit_e, lane_strip = (np.asarray(region(impl)) for region in
                                       (ROAD_E, ENTRY_W, EXIT_E, BoxSet(dict(y=LANE_STRIP), axes=['y'])))

# Fidelity: the same inside sets as the legacy map code, at every time sample.
legacy = create_4way(grid, 'road_e', 'entry_w', 'exit_e')
assert all(np.array_equal(region <= 0, np.broadcast_to(legacy[name] <= 0, region.shape))
           for name, region in (('road_e', road_e), ('entry_w', entry_w), ('exit_e', exit_e)))

**Generator [numerical].** The owner's heuristic for candidate $R_\kappa$; not evidence.
pass 1 = BRS(`exit_e` ∣ `road_e`); pass 2 = pass 1 (no reservations); pass 3 = FRS(`entry_w` ∩ pass 2);
pass 4 = BRS(pass 3 ∩ `exit_e`); corridor = pass 4 ∩ lane strip.

In [ ]:
analysis = planner.analyze(entry=entry_w, goal=exit_e, constraints=road_e,
                           entry_window=TimeWindow(0, .5), exit_window=TimeWindow(7, 8),
                           policy=WindowPolicy(0, .5, 0, 1))
passes = analysis.passes
assert passes['pass2'] is passes['pass1']
corridor = np.maximum(passes['pass4'], lane_strip).astype(np.float32)

steps = {'pass1': 'backward reach of exit_e under road_e', 'pass2': 'pass 1 itself (no reservations)',
         'pass3': f"forward from entry_w ∩ pass 2, entry window [{analysis.entry_window.start:g}, {analysis.entry_window.end:g}] s",
         'pass4': f"backward from pass 3 ∩ exit_e, exit window [{analysis.exit_window.start:g}, {analysis.exit_window.end:g}] s",
         'corridor': f'pass 4 ∩ lane strip y ∈ {LANE_STRIP}'}
tubes = dict(passes, corridor=corridor)
entry = tuple(int(np.abs(c - e).argmin()) for c, e in zip((x, y, h, v), ENTRY))  # grid index of z0
table(['Tube', 'Restriction', 'Inside fraction', 'Value at entry'],
      [(name, step, f'{(tubes[name] <= 0).mean():.4f}', f'{tubes[name][(0, *entry)]:.4f}') for name, step in steps.items()])

**Reservation $R_\kappa$ [numerical].** Project over $(h,v)$ per sample; inflate by half a cell plus
one slab's travel; slab $i$ also covers $t_{i+1}$.

In [ ]:
occupied = (corridor <= 0).any(axis=(3, 4))  # (t, x, y)
dx, dy = x[1] - x[0], y[1] - y[0]
margin = (dx/2 + VMAX*DT, dy/2 + VMAX*np.sin(CHART['h'][1])*DT)
kx, ky = int(np.ceil(margin[0]/dx)), int(np.ceil(margin[1]/dy))
padded = np.pad(occupied, ((0, 0), (kx+2, kx+2), (ky+2, ky+2)))  # room for margins at the domain edge
slabs = padded.copy()
slabs[:-1] |= padded[1:]
reserved = maximum_filter(slabs.astype(np.uint8), size=(1, 2*kx+1, 2*ky+1), mode='constant') > 0
rx = x[0] + np.arange(-(kx+2), len(x)+kx+2)*dx
ry = y[0] + np.arange(-(ky+2), len(y)+ky+2)*dy

table(['Reservation', 'Value'], [('Margins (x, y) [m]', margin), ('Reserved cells', int(reserved.sum()))])

**Query generated from $F_s$ [numerical].** For a template at $s$, with $\mathit{alloc}:=R_\kappa$:
$$z_0\in\mathrm{Sat}_\exists\big((\mathit{road}\land R_\kappa)\ \mathcal U_{[s,s+8]}\ \mathit{exit}\big)\ \text{at }s
\ \Rightarrow\ F_s(R_\kappa)\ \Leftrightarrow\ H_\kappa\vDash_f F_s\qquad(s+8\le d_\theta).$$
- Time-invariant dynamics and the π-symmetric map: one query at $s=0$ covers every shift and rotation.
- `compile_query`: state predicates → one realized set; bounded until → TLT until with a box on $t$.

In [ ]:
def allocation(reserved, impl):
    # alloc := R as a level set on impl's grid: signed distance [m] to the reserved cells of the
    # slab containing each time sample (slab i covers [t_i, t_i+1]), constant in (h, v).
    cells = reserved[:, kx+2:kx+2+len(x), ky+2:ky+2+len(y)]  # the reservation raster on the HJ grid
    values = np.ones(cells.shape, np.float32)
    for k, c in enumerate(cells):
        if c.any():
            values[k] = np.where(c, -distance_transform_edt(c, sampling=(dx, dy)),
                                 distance_transform_edt(~c, sampling=(dx, dy)))
    slab = np.minimum((np.asarray(impl.timeline)/DT + 1e-6).astype(int), len(cells) - 1)
    return np.broadcast_to(values[slab][..., None, None], impl.shape)


template = CrossingService('X', 'east', (0,), LEASE)  # one template at s = 0: window [0, 8]
query = template.task_for(0)                            # Φ_0, the interpretation of F_0
INTERPRETATION = {template.road.name: ROAD_E, template.exit.name: EXIT_E}
tree = compile_query(query, 0, INTERPRETATION | {template.alloc.name: Set(allocation(reserved, impl))})
capability = np.asarray(tree.realize(impl))  # V_R
TOLERANCE = 2e-5
table(['Generated query', 'Value'], [
    ('Owner obligation', template.access_for(0)), ('Interpreted formula $\\Phi_0$', query),
    ('Interpretation', 'road, exit: §2 set builders; alloc: $R_\\kappa$ as a signed-distance level set'),
    ('TLT', f'`{tree!r}`'),
    ('$V_R$ at the entry state', f'{capability[(0, *entry)]:.5f} (inside if ≤ {TOLERANCE})')], template.symbols)
assert capability[(0, *entry)] <= TOLERANCE

**Declared client [independent].** $\pi_R$ from $V_R$, executed from $z_0$; each sample checked against
`LANE` ⊆ `road_e`, the closed reserved cells of its slab, and `LANE` ∩ `EXIT_ZONE` ⊆ `exit_e`.

In [ ]:
grids = (times, x, y, h, v)
slopes = [RegularGridInterpolator(grids, np.gradient(capability, c, axis=i), bounds_error=False, fill_value=None)
          for i, c in ((3, h), (4, v))]
value = RegularGridInterpolator(grids, capability, bounds_error=False, fill_value=None)
policy = hj_feedback(lambda t, z: tuple(float(slope([[t, *z]])[0]) for slope in slopes))
taus, path, inputs = execute(policy)


def executed_in(reserved, delay=0.):
    # Every execution sample, `delay` after the reservation's time zero, lies in closed reserved cells.
    slab = ((taus + delay)/DT + 1e-6).astype(int)
    return bool(slab[-1] < len(reserved)) and all(
        boxes_covered(reserved, rx, ry, [(k, p[0], p[0], p[1], p[1])]) for k, p in zip(slab, path))


SERVICE_EVIDENCE = dict(
    capability=bool(capability[(0, *entry)] <= TOLERANCE),
    policy_on_road=all(inside(z, LANE) for z in path),
    policy_in_allocation=executed_in(reserved),
    policy_reaches_exit=bool(inside(path[-1], LANE, EXIT_ZONE) and taus[-1] <= LEASE),
    policy_inputs_bounded=bool(np.all((inputs >= [CONTROLS['min_steer'], CONTROLS['min_accel']])
                                      & (inputs <= [CONTROLS['max_steer'], CONTROLS['max_accel']]))))
table(['Execution', 'Value'], [('Exit reached at [s]', f'{taus[-1]:.3f}'),
                               ('Max $V_R$ along the execution', f'{value(np.column_stack([taus, path])).max():.5f}'),
                               ('y range [m]', (path[:, 1].min(), path[:, 1].max())),
                               *SERVICE_EVIDENCE.items()])
assert all(SERVICE_EVIDENCE.values())

base_path = OUTPUT / 'base.npz'  # the full corridor every offer artifact refers to
np.savez_compressed(base_path, tube=corridor, reserved=reserved, rx=rx, ry=ry, times=times, capability=capability)
BASE = dict(reserved=reserved, dt=DT, path=str(base_path), digest=file_digest(base_path),
            service_checks=SERVICE_EVIDENCE, output=str(OUTPUT))

- Generator: passes, strip and projection construct $R_\kappa$.
- Query: generated from $F_s$; decides $H_\kappa\vDash_f F_s$.
- Client: $\pi_R$ executes it; the execution checks $\Phi_0$ independently.

## 3. Contracts, admission and offers

$$C_{\rm client}=\langle S_a\mid T_\theta\rangle,\quad R_{\rm task}=\langle\top\mid T_\theta\rangle,$$
$$Q=R_{\rm task}/C_{\rm client}=\langle\neg S_a\lor T_\theta\mid S_a\rangle,\quad
C_{\rm cmt}^{+}=C_{\rm cmt}\wedge Q=\langle\top\mid L\land S_a\rangle .$$
- Services in order; an adaptation rebuilds client, task and $Q$.
- Template: base reservation shifted to $s$, rotated to the route. Service and compatibility
  evidence together witness $L\land S_a$ (sufficient, finite family).
- Statuses: `QUALIFIED` · `GEOMETRIC_CONFLICT` · `UNRESOLVED` ($s+8>d_\theta$: certified window does not fit) ·
  `PROVED_IMPOSSIBLE` (speed bound refutes $F_s$).

In [ ]:
def service_contracts(service):
    # III-A: the fixed client and the joint task for one selected service.
    S_a, T = il.Or(service.access_for(s) for s in service.departures), service.task
    return S_a, T, Contract(S_a, T), Contract(True, T)


def route_reservation(base, route):
    # Westbound is the eastbound map rotated by π: reverse both workspace axes.
    return base['reserved'] if route == 'east' else base['reserved'][:, ::-1, ::-1]


def qualify(service, departure, snapshot, ledger, base):
    # One finite template, checked against every standing reservation of the frozen snapshot.
    offer_id = digest(dict(service=service.key, departure=departure, snapshot=snapshot,
                           base=base['digest']))[:20]
    if speed_bound_excludes(departure, service.deadline):
        return Candidate(departure, offer_id, dict(
            status='PROVED_IMPOSSIBLE', evidence='independent analytic bound',
            reason=f"{EXIT_ZONE['x'][0] - ENTRY[0]:g} m to the exit exceeds vmax · {service.deadline - departure} s"))
    if not certified_window_applies(departure, service.deadline):
        return Candidate(departure, offer_id, dict(
            status='UNRESOLVED', evidence='numerical evidence does not apply',
            reason='certified window [s, s+8] is not within the deadline'))
    mine = route_reservation(base, service.route)
    comparisons = [dict(offer=b.offer.id, disjoint=aligned_disjoint(
                       mine, departure, route_reservation(base, b.offer.service.route),
                       b.offer.departure, base['dt']))
                   for b in ledger]
    if not all(c['disjoint'] for c in comparisons):
        return Candidate(departure, offer_id, dict(
            status='GEOMETRIC_CONFLICT', evidence='template rejected', reason='reservation meets a standing one',
            service=base['service_checks'], comparisons=comparisons))
    info = dict(id=offer_id, service=asdict(service), departure=departure, route=service.route,
                snapshot=snapshot, end=departure + LEASE, base=base['path'], base_digest=base['digest'],
                checks=dict(service={**base['service_checks'], 'window_within_deadline': True},
                            compatibility=dict(compatible=True, disjoint_from=comparisons)))
    artifact, artifact_digest = write_artifact(base['output'], info)
    return Candidate(departure, offer_id, dict(status='QUALIFIED', artifact=artifact,
                                               artifact_digest=artifact_digest, info=info))


def admit(request, ledger, base):
    # Frozen snapshot: the standing commitments C_cmt = <T | L> stay fixed for this attempt.
    ledger = tuple(ledger)
    commitment, snapshot = standing(ledger), snapshot_id(ledger)
    tried = []
    for service in request.services:  # the original service, then permitted adaptations
        S_a, T, C_client, R_task = service_contracts(service)

        # III-B, owner obligations: Q and its access-only guarantee.
        Q = R_task.quotient(C_client).normalize()
        assert Q.assumption == normalize(~S_a | T)
        assert Q.guarantee == normalize(S_a)

        # III-B, retain standing obligations: C_plus = C_cmt ∧ Q = <T | L ∧ S_a>.
        C_plus = commitment.conjoin(Q)
        assert normalize(C_plus.guarantee) == normalize(commitment.guarantee & S_a)

        # Address L ∧ S_a within the finite family, template by template.
        candidates = tuple(qualify(service, s, snapshot, ledger, base) for s in service.departures)
        tried.append(ServiceAdmission(service, S_a, T, C_client, R_task, Q, C_plus, candidates))
        if any(c.status == 'QUALIFIED' for c in candidates):
            break  # freeze ONE service and client model before generating offers
    return Admission(request, snapshot, commitment, tuple(tried))

$$z_0\in\mathrm{Sat}_\exists\big((\mathit{road}\land R_\kappa)\,\mathcal U_{[s,s+8]}\,\mathit{exit}\big)\ \Rightarrow\
H_\kappa\vDash_f F_s\Rightarrow S_a,\qquad K_\kappa=\langle\top\mid H_\kappa\land F_s\rangle .$$
- $K_\kappa\Rightarrow S_a$ is algebraic; that $F_s$ is redundant given $H_\kappa$ is the capability evidence, checked first.
- Compatibility: disjoint reservations, so $r_i:=R_{\kappa_i}$ satisfies $L\land H_\kappa$; checked separately.
- **[algebra]** $K_\kappa\preceq Q$, $C_{\rm cmt}\wedge K_\kappa\preceq C_{\rm cmt}^{+}$,
  $C_{\rm client}\otimes(C_{\rm cmt}\wedge K_\kappa)\preceq C_{\rm cmt}\wedge R_{\rm task}$.

In [ ]:
def generate(admission):
    # III-C: one offer per qualified template of the single selected service.
    selected = admission.selected
    offers = []
    for candidate in (() if selected is None else selected.qualified):
        info = candidate.reply['info']
        offer = CorridorOffer(candidate.offer_id, selected.service, selected.S_a, candidate.departure,
                              F(info['end']), admission.snapshot, candidate.reply['artifact'],
                              candidate.reply['artifact_digest'], json.dumps(info['checks'], sort_keys=True))
        certify_service(offer)        # capability evidence first, then K_kappa ⇒ S_a
        certify_compatibility(offer)  # disjoint from every standing reservation
        offers.append(offer)
    return tuple(offers)  # frozen: one service, one client model, one snapshot

- `bargain`: verbatim from `protocol.ipynb`; the validator requires both certificates and one
  service, client model and snapshot.
- Commit: exact object and price, $C_{\rm cmt}\leftarrow C_{\rm cmt}\wedge K^*$. No agreement keeps
  commitments and the failure reason.

In [ ]:
def commit(admission, offers, negotiation, ledger):
    ledger = tuple(ledger)
    if negotiation.agreement is None:
        # III-D: no agreement keeps every commitment; the outcome keeps its reason.
        status = 'PRICE-NO-BLA' if offers else admission.failure
        return CommitResult(status, admission.commitment, ledger, None)
    # Exact accepted object, fresh snapshot, both certificates and unchanged artifacts.
    updated = commit_exact(offers, negotiation, ledger, verify_artifact)
    return CommitResult('COMMITTED', admission.commitment.conjoin(updated[-1].contract),
                        updated, updated[-1])


def attempt(spec, ledger, base):
    # admit → generate → bargain → commit, unchanged, for one request.
    request, previous = spec['request'], tuple(ledger)
    for bla in previous:
        verify_artifact(bla.offer)
    admission = admit(request, previous, base)
    offers = generate(admission)
    policies = affine_policies(dict(zip(offers, spec['costs'])), dict(zip(offers, spec['values'])))
    negotiation = bargain(offers, policies, spec['targets'], spec['minima'], PRICES, ROUND_BUDGET)
    result = commit(admission, offers, negotiation, previous)
    # Every earlier BLA object and artifact is retained, whatever the outcome.
    preserved = len(result.ledger) >= len(previous) and all(a is b for a, b in zip(result.ledger, previous))
    for bla in previous:
        verify_artifact(bla.offer)
    assert preserved and (result.status == 'COMMITTED' or result.ledger is previous)
    return dict(request=request, admission=admission, offers=offers, policies=policies,
                negotiation=negotiation, result=result, previous_preserved=preserved)

## 4. Staged execution

| Client | Case |
|:--|:--|
| A | first eastbound crossing |
| B | two westbound alternatives during A's lease |
| C | same lane as A: conflict |
| D | price failure |
| E | original deadline impossible; permitted later service |

Costs/values: earlier offer 5/10, later 2/8; D values 1/0.

In [ ]:
PRICES, ROUND_BUDGET = (F(0), F(20)), 3
PREFERENCES = dict(costs=(F(5), F(2)), values=(F(10), F(8)),
                   targets={'owner': F(4), 'client': F(4)}, minima={'owner': F(1), 'client': F(1)})
PRICE_FAILURE = dict(costs=(F(5), F(2)), values=(F(1), F(0)),
                     targets={'owner': F(4), 'client': F(1)}, minima={'owner': F(2), 'client': F(0)})
REQUESTS = (
    dict(request=CrossingRequest('A', CrossingService('A', 'east', (0,), 8)), **PREFERENCES),
    dict(request=CrossingRequest('B', CrossingService('B', 'west', (0, 1), 9)), **PREFERENCES),
    dict(request=CrossingRequest('C', CrossingService('C', 'east', (0, 1), 9)), **PREFERENCES),
    dict(request=CrossingRequest('D', CrossingService('D', 'west', (10, 11), 19)), **PRICE_FAILURE),
    dict(request=CrossingRequest('E', CrossingService('E', 'east', (20,), 22),
                                 alternatives=(CrossingService('E', 'east', (20,), 28),)), **PREFERENCES),
)

In [ ]:
table(['Service', 'Departure s', 'Deadline d', 'Certified window applies: s+8 ≤ d',
       'Speed bound refutes $F_s$ [independent]'],
      [(f'{svc.client} {svc.route}', s, svc.deadline,
        certified_window_applies(s, svc.deadline), speed_bound_excludes(s, svc.deadline))
       for spec in REQUESTS for svc in spec['request'].services for s in svc.departures])

### A: initial commitment

In [ ]:
ledger, attempts = (), []
A = attempt(REQUESTS[0], ledger, BASE)
attempts.append(A)
ledger = A['result'].ledger
assert A['result'].status == 'COMMITTED' and len(ledger) == 1
names = {b.offer.id: b.offer.service.client for b in ledger}
table(['Client', 'Outcome', 'Accepted contract $K^*$', 'Price', 'Lease [s]', '$C_{\\rm cmt}$ after A'],
      [('A', A['result'].status, ledger[-1].contract, ledger[-1].price,
        f'[{ledger[-1].offer.departure}, {ledger[-1].offer.end}]', A['result'].commitment)],
      ledger_symbols(ledger))

### B.1 Admission against A's snapshot

In [ ]:
B_spec = REQUESTS[1]
frozen = ledger  # A's BLA; fixed for B's whole attempt
admission = admit(B_spec['request'], frozen, BASE)
selected = admission.selected
symbols = ledger_symbols(frozen) | selected.service.symbols
table(['Object [algebra]', 'Value'], [
    ('$S_a$', selected.S_a), ('$T_\theta$', selected.T),
    (r'$C_{\rm client}$', selected.client), (r'$R_{\rm task}$', selected.task), ('$Q$', selected.quotient),
    (r'$C_{\rm cmt}$ (A)', admission.commitment), (r'$C_{\rm cmt}^{+}$', selected.obligations)], symbols)
assert selected.quotient.assumption == normalize(~selected.S_a | selected.T)
assert selected.quotient.guarantee == normalize(selected.S_a)
assert normalize(selected.obligations.guarantee) == normalize(admission.commitment.guarantee & selected.S_a)
assert admission.snapshot == snapshot_id(frozen) and admission.commitment == standing(frozen)

In [ ]:
table(['Service', 'Departure', 'Status', 'Meaning', 'Service evidence', 'Compatibility', 'Reason'],
      list(candidate_rows(admission, names)))
assert [c.status for c in selected.candidates] == ['QUALIFIED', 'QUALIFIED']

### B.2 Offers: capability evidence, then $K_\kappa\Rightarrow S_a$; compatibility

In [ ]:
offers = generate(admission)
assert len(offers) == 2 and all(o.service is selected.service and o.access == selected.S_a
                                and o.snapshot == admission.snapshot for o in offers)
symbols |= ledger_symbols((), *offers)
label = lambda o: f'κ(s={o.departure})'
table(['Offer', 'Lease [s]', 'Query: $z_0$ inside', '$\\pi_R$ on road', '$\\pi_R$ in $R_\\kappa$',
       '$\\pi_R$ reaches exit', 'Inputs bounded', 'Window ⊆ [s, d]'],
      [(label(o), f'[{o.departure}, {o.end}]', *(o.evidence['service'][k] for k in SERVICE_CHECKS))
       for o in offers])
table(['Offer', '$K_\\kappa$', '$K_\\kappa\\Rightarrow S_a$ [algebra]'],
      [(label(o), o.contract, implies(o.contract.guarantee, selected.S_a)) for o in offers], symbols)

In [ ]:
def composition_checks(offer, admission, selected):
    C_cmt, K = admission.commitment, offer.contract
    return dict(K_refines_Q=refines(K, selected.quotient),
                conjunction=refines(C_cmt.conjoin(K), selected.obligations),
                composition=refines(selected.client.compose(C_cmt.conjoin(K)), C_cmt.conjoin(selected.task)))

table(['Offer', 'Disjoint from [numerical]', 'Allocation witness for $L\\land H_\\kappa$ [semantic]',
       '$K_\\kappa\\preceq Q$', '$C_{\\rm cmt}\\wedge K_\\kappa\\preceq C_{\\rm cmt}^{+}$',
       '$C_{\\rm client}\\otimes(C_{\\rm cmt}\\wedge K_\\kappa)\\preceq C_{\\rm cmt}\\wedge R_{\\rm task}$'],
      [(label(o), ', '.join(f"{names[c['offer']]}: {c['disjoint']}" for c in o.evidence['compatibility']['disjoint_from']),
        'r_A := R_A, r_B := R_κ', *composition_checks(o, admission, selected).values()) for o in offers])
assert all(all(composition_checks(o, admission, selected).values()) for o in offers)

### B.3 Bargaining

In [ ]:
policies = affine_policies(dict(zip(offers, B_spec['costs'])), dict(zip(offers, B_spec['values'])))
table(['Offer', 'Cost $c_\\kappa$', 'Value $v_\\kappa$', 'Targets (owner, client)', 'Minima (owner, client)'],
      [(label(o), B_spec['costs'][i], B_spec['values'][i], tuple(B_spec['targets'].values()),
        tuple(B_spec['minima'].values())) for i, o in enumerate(offers)])
negotiation = bargain(offers, policies, B_spec['targets'], B_spec['minima'], PRICES, ROUND_BUDGET)
table(['Round', 'Proposer', 'Targets (owner, client)', 'Offer', 'Price', 'Utilities (owner, client)', 'Accepted'],
      [(r.number, r.proposer, tuple(v for _, v in r.targets), label(r.proposal.offer), r.proposal.price,
        tuple(v for _, v in r.utilities), r.accepted) for r in negotiation.history])
assert negotiation.status == 'ACCEPTED' and negotiation.offers == offers

### B.4 Commitment

In [ ]:
result = commit(admission, offers, negotiation, frozen)
service = selected.service
# Fixed client rule: the earliest s whose F_s the commitment implies.
departure = min(s for s in service.departures if implies(result.commitment.guarantee, service.access_for(s)))
try:
    commit(admission, offers, negotiation, result.ledger)
    stale = 'accepted'
except ValueError as error:
    stale = f'rejected: {error}'
table(['Commitment item', 'Value'], [
    ('Status', result.status), ('Accepted offer', label(result.agreement.offer)), ('Price', result.agreement.price),
    ('Exact object stored', result.agreement.offer is negotiation.agreement.offer),
    ("A's BLA object retained", result.ledger[0] is frozen[0]),
    ('$C_{\\rm cmt}$ after B', result.commitment),
    ('Departure chosen by the fixed client rule', departure),
    ('Re-committing against the updated snapshot', stale)], ledger_symbols(result.ledger))
assert result.status == 'COMMITTED' and departure == result.agreement.offer.departure
assert stale.startswith('rejected')
B = dict(request=B_spec['request'], admission=admission, offers=offers, policies=policies,
         negotiation=negotiation, result=result,
         previous_preserved=all(a is b for a, b in zip(result.ledger, frozen)))
attempts.append(B)
ledger = result.ledger
names.update({b.offer.id: b.offer.service.client for b in ledger})

### C–E

In [ ]:
for spec in REQUESTS[2:]:
    record = attempt(spec, ledger, BASE)
    attempts.append(record)
    ledger = record['result'].ledger
    names.update({b.offer.id: b.offer.service.client for b in ledger})
C, D, E = attempts[2:]
table(['Service', 'Departure', 'Status', 'Meaning', 'Service evidence', 'Compatibility', 'Reason'],
      [row for a in (C, D, E) for row in candidate_rows(a['admission'], names)])
table(['E service', '$S_a$', '$Q$', 'Templates'],
      [(f'd={s.service.deadline}', s.S_a, s.quotient, ', '.join(c.status for c in s.candidates))
       for s in E['admission'].services], E['admission'].services[-1].service.symbols)
table(['Client', 'Round', 'Proposer', 'Offer', 'Price', 'Utilities (owner, client)', 'Accepted'],
      [(a['request'].client, r.number, r.proposer, label(r.proposal.offer), r.proposal.price,
        tuple(v for _, v in r.utilities), r.accepted) for a in (C, D, E) for r in a['negotiation'].history])
table(['Client', 'Outcome', 'Meaning', 'Earlier BLAs retained', 'BLAs after'],
      [(a['request'].client, a['result'].status, MEANINGS[a['result'].status], a['previous_preserved'],
        len(a['result'].ledger)) for a in attempts])

## 5. Evidence

Accepted behaviour under $r_j:=R_\kappa$ and the declared client's execution at the accepted $s$.

In [ ]:
def evidence_values(ledger):
    # Allocation witness: all accepted reservations pairwise disjoint (each offer was checked
    # against exactly the BLAs before it), so r_j := R_kappa_j is one allocation.
    values, witness = {}, True
    for i, bla in enumerate(ledger):
        o, e = bla.offer, bla.offer.evidence['service']
        witness &= ({c['offer'] for c in bla.offer.evidence['compatibility']['disjoint_from']}
                    == {b.offer.id for b in ledger[:i]})
        executed = all(e[k] for k in ('policy_on_road', 'policy_in_allocation', 'policy_reaches_exit',
                                      'policy_inputs_bounded'))
        values[o.promise] = witness
        values[o.service.access_for(o.departure)] = e['capability']
        for s in o.service.departures:  # Φ_s holds for the departure the client executed
            values[o.service.task_for(s)] = executed and s == o.departure
    return values


def holds(formula, values):
    # The Boolean skeleton; atoms and until subformulas take their evidence values.
    f = il.as_formula(formula)
    if isinstance(f, il.Boolean):
        return f.value
    if isinstance(f, (il.Proposition, il.Until)):
        return values.get(f, False)
    parts = [holds(c, values) for c in f.children]
    if isinstance(f, il.Not):
        return not parts[0]
    if isinstance(f, il.And):
        return all(parts)
    if isinstance(f, il.Or):
        return any(parts)
    raise ValueError(f'not in the Boolean skeleton: {f}')


values, L = evidence_values(ledger), standing(ledger).guarantee
behaviour = {}
for bla in ledger:
    o = bla.offer
    S_a, T, C_client, R_task = service_contracts(o.service)
    behaviour[o.service.client] = dict(
        owner=holds(L, values), access=holds(o.service.access_for(o.departure), values),
        assumption=holds(S_a, values), guarantee=holds(C_client.guarantee, values),
        task=holds(R_task.guarantee, values))
evidence = {'owner': '[semantic + numerical] disjoint reservations; r := R is one allocation',
            'access': '[numerical] generated query with alloc := R_κ; H_κ ⊨ F_s',
            'assumption': '[algebra] F_s ⇒ S_a (normalization)',
            'guarantee': '[semantic + independent] π_R executes Φ_s: on road, in R_κ, reaches exit',
            'task': '[independent] Φ_s at the accepted s ⇒ T_θ'}
rows = {'owner': 'Owner promise $L$', 'access': 'Capability $F_s$', 'assumption': 'Client assumption $S_a$',
        'guarantee': 'Client guarantee $T_\\theta\\lor\\neg S_a$', 'task': 'Task $T_\\theta$'}
table(['Accepted behaviour', *behaviour, 'Evidence'],
      [(rows[k], *(b[k] for b in behaviour.values()), evidence[k]) for k in rows])
assert all(all(b.values()) for b in behaviour.values())

**Independent checks:** pairwise separation of the executions; the generated query at half the time step.

In [ ]:
def physical_path(offer):
    states = path.copy()
    if offer.service.route == 'west':  # rotation by π
        states[:, :2] *= -1
        states[:, 2] += np.pi
    return taus + offer.departure, states

pair_checks = []
for a, b in combinations(ledger, 2):
    gap = sampled_separation(*physical_path(a.offer), *physical_path(b.offer), .025)
    pair_checks.append(dict(first=a.offer.id, second=b.offer.id, minimum_separation=None if np.isinf(gap) else gap))
gaps = [p['minimum_separation'] for p in pair_checks if p['minimum_separation'] is not None]

FINE_AXES = [dict(AXES[0], points=round(HORIZON/(DT/2)) + 1), *AXES[1:]]
fine = LTMSPlanner(TVHJImpl(dict(DYNAMICS), FINE_AXES, accuracy=ACCURACY))
fine_tree = compile_query(query, 0, INTERPRETATION | {template.alloc.name: Set(allocation(reserved, fine.impl))})
fine_entry = float(np.asarray(fine_tree.realize(fine.impl))[(0, *entry)])  # the same query, half the step
assert gaps and min(gaps) > 0 and fine_entry <= TOLERANCE
table(['Check or convention', 'Value'], [
    ('HJ membership', f'value ≤ {TOLERANCE} on the declared grid; time step {DT} s'),
    ('Temporal refinement [numerical]',
     f"time step {DT/2} s: $V_R$ at entry {capability[(0, *entry)]:.5f} → {fine_entry:.5f}"),
    ('Client execution', 'RK4, input held per 0.025 s step; checks at every sample'),
    ('Reservation contact', 'closed raster cells; neighbouring time slabs also compared'),
    ('Simultaneous vehicles [independent]', f'minimum pairwise separation, sampled every 0.025 s: {min(gaps):.2f} m'),
    ('Leases', f'allocations stay promised on [s, s+8]; the execution exits at s+{taus[-1]:.2f}')])

**Failure meanings:** an empty family is not impossibility; `UNRESOLVED` is neither an offer nor a proof.

In [ ]:
# Geometric conflict: C's template is certified for service, Boolean-satisfiable with L, yet meets A.
conflict = C['admission'].services[0].candidates[0]
H_C = il.Proposition(f'H_{conflict.offer_id}')
F_C = C['admission'].services[0].service.access_for(conflict.departure)
boolean_sat = normalize(C['admission'].commitment.guarantee & H_C & F_C) != il.Boolean(False)

# Missing capability: the generated query answers no when the allocation starts 1 s late, the
# declared execution leaves reservations delayed by 1 s / 2 s, and an offer without capability
# evidence is rejected before any implication is used.
late = np.concatenate([np.zeros_like(reserved[:4]), reserved[:-4]])
late_tree = compile_query(query, 0, INTERPRETATION | {template.alloc.name: Set(allocation(late, impl))})
late_value = float(np.asarray(late_tree.realize(impl))[(0, *entry)])
shifted = {delay: executed_in(reserved, delay) for delay in (1., 2.)}
first = ledger[0].offer
lacking = replace(first, evidence_json=json.dumps(
    {**first.evidence, 'service': {**first.evidence['service'], 'capability': False}}))
try:
    certify_service(lacking)
    capability_rejected = False
except ValueError:
    capability_rejected = True

# Unresolved: F_30 is in S_a, but the certified window [s, s+8] exceeds d.
boundary = admit(CrossingRequest('F', CrossingService('F', 'east', (30,), 37)), ledger, BASE)
impossible = E['admission'].services[0].candidates[0]

table(['Case', 'Evidence', 'Meaning', 'Consequence'], [
    ('C: geometric conflict', f"service evidence {all(conflict.reply['service'].values())}; "
     f"$L\\land H_\\kappa\\land F_s$ Boolean-satisfiable: {boolean_sat}; reservations meet A's",
     MEANINGS['GEOMETRIC_CONFLICT'], 'empty family: NO-COMPATIBLE-OFFER, not impossibility'),
    ('Missing capability', f"query with R_κ 1 s late: $V$ at entry {late_value:.3f}; execution in R_κ "
     f"delayed 1 s / 2 s: {shifted[1.]} / {shifted[2.]}; offer without capability evidence rejected: "
     f"{capability_rejected}", 'H_κ would not entail F_s', 'cannot qualify; the Boolean implication is never consulted'),
    ('F (s=30, d=37): unresolved', f"certified window applies {certified_window_applies(30, 37)}; "
     f"status {boundary.services[0].candidates[0].status}",
     MEANINGS['UNRESOLVED'], f"admission failure {boundary.failure}; nothing committed"),
    ('E original (d=22)', impossible.reply['reason'], MEANINGS['PROVED_IMPOSSIBLE'],
     'permitted replacement d=28 admitted with rebuilt contracts'),
    ('D: price failure', f"{len(D['offers'])} qualified offers; {len(D['negotiation'].history)} rejected rounds",
     MEANINGS['PRICE-NO-BLA'], 'commitments unchanged')])
assert conflict.status == 'GEOMETRIC_CONFLICT' and boolean_sat and all(conflict.reply['service'].values())
assert late_value > TOLERANCE and not shifted[2.] and capability_rejected
assert boundary.selected is None and boundary.failure == 'UNRESOLVED'
assert impossible.status == 'PROVED_IMPOSSIBLE' and E['admission'].selected.service.deadline == 28

Shading: time union of reservations (display only). Right panel: $h=0$, $v=0.45$.

In [ ]:
final = dict(
    paths=[dict(id=b.offer.id, route=b.offer.service.route,
                times=physical_path(b.offer)[0].tolist(), states=physical_path(b.offer)[1].tolist(),
                reserved_union=route_reservation(BASE, b.offer.service.route).any(axis=0).T.astype(int).tolist())
           for b in ledger],
    pair_checks=pair_checks, x=x.tolist(), y=y.tolist(), rx=rx.tolist(), ry=ry.tolist(),
    slice=capability[0, :, :, entry[2], entry[3]].T.tolist())
numerics = dict(axes=AXES, accuracy=ACCURACY, base=dict(path=BASE['path'], digest=BASE['digest']),
                capability_entry=float(capability[(0, *entry)]), refinement_entry=fine_entry, service_evidence=SERVICE_EVIDENCE)
report = build_report(ROOT, attempts, ledger, final, numerics)
display(Markdown(results_table(report)))
figure = plot_scenario(report)
display_timeline(figure)
export_report(report, figure, OUTPUT)
export_figure(final, report['attempts'], OUTPUT)

**Runner agreement:** `run_scenario` must reproduce outcomes, offers, departures, prices and retained promises.

In [ ]:
runner_attempts, runner_ledger = run_scenario(ROOT, BASE)
assert outcome_summary(runner_attempts) == outcome_summary(attempts)
for bla in ledger:
    verify_artifact(bla.offer)
assert [a['result'].status for a in attempts] == [
    'COMMITTED', 'COMMITTED', 'NO-COMPATIBLE-OFFER', 'PRICE-NO-BLA', 'COMMITTED']
assert all(a['previous_preserved'] for a in attempts)
assert all(a['result'].agreement.offer is a['negotiation'].agreement.offer
           for a in attempts if a['result'].agreement)
assert [b.offer.service.client for b in ledger] == ['A', 'B', 'E']
table(['Client', 'Outcome', 'Offers', 'Selected departure', 'Price', 'Retained BLAs'],
      [(name, status, len(ids), '—' if s is None else s, '—' if q is None else q, len(kept))
       for name, status, ids, s, q, kept in outcome_summary(runner_attempts)])

## Reproduction

```sh
uv run --group notebooks python scripts/run_ltms_notebook.py --save
uv run --group notebooks python scripts/run_ltms_notebook.py --output /tmp/ltms  # elsewhere
```
Artifacts (ignored by Git) go to `artifacts/ltms_blas/`.